Compare encoding performance for sentence start and end words for banded ridge models

In [1]:
import pandas as pd
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import pearsonr

In [2]:
ind_d = "/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/results/misc/sentence_word_indices"
elecs_perf_comp_f = f"{ind_d}/comprehension_future_good_electrodes.csv"
elecs_perf_prod_f = f"{ind_d}/production_future_good_electrodes.csv"
preds_d_tmp = "/scratch/gpfs/HASSON/ij9216/projects/code/247/247-encoding-dev/results/tfs/ij-tfs-{sid}-gpt2-xl-bandedRidge-lag2k-50-all-static_future_past-no-reph_pca_drop-short_mistral/ij-200ms-{sid}/"

# There are electrodes with missing convos that have different sample sizes - need to extract indices for them separately
max_dim1_dict = {
    ('625', 'comp'): 38176,
    ('625', 'prod'): 24511,
    ('676', 'comp'): 91202,
    ('676', 'prod'): 92442,
    ('7170', 'comp'): 58869,
    ('7170', 'prod'): 38564,
    ('798', 'comp'): 50308,
    ('798', 'prod'): 40869,
}
# file format 798_DPMT1_comp_predictions.h5  

elecs_comp = pd.read_csv(elecs_perf_comp_f)
elecs_prod = pd.read_csv(elecs_perf_prod_f)

output_pdf = f"{ind_d}/sentence_start_end_encoding_comparison.pdf"
weird_lengths_csv = f"{ind_d}/weird_length_electrodes.csv"

# Lag values for x-axis (81 values from -2000ms to 2000ms in 50ms steps)
lags_ms = np.arange(-2000, 2001, 50)
lags_s = lags_ms / 1000  # Convert to seconds for plotting

# Get default matplotlib colors (matching tfsplt_future_past_utils.py convention)
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
# Color mapping: colors[0]=joint, colors[1]=word, colors[2]=future/sentence, colors[3]=past/sentence2
pred_type_colors = {
    'joint': default_colors[0],
    'word': default_colors[1],
    'future': default_colors[2],
    'past': default_colors[3]
}

def calculate_correlations_per_lag(predictions, indices=None):
    """
    Calculate correlations between dim 0 (actual) and dims 1-4 (joint, word, future, past)
    across all lag values (dim3)
    predictions: shape (5, n_timepoints, n_lags)
    Order: actual, joint, word, future, past
    indices: optional subset of timepoint indices
    Returns: dict with keys 'joint', 'word', 'future', 'past' containing arrays of shape (n_lags,)
    """
    if indices is not None:
        preds_subset = predictions[:, indices, :]
    else:
        preds_subset = predictions
    
    # Get predictions from each dimension (5, n_samples, n_lags)
    dim0 = preds_subset[0]  # actual
    dim1 = preds_subset[1]  # joint
    dim2 = preds_subset[2]  # word
    dim3 = preds_subset[3]  # future
    dim4 = preds_subset[4]  # past
    
    # Calculate correlations across timepoints for each lag
    n_lags = dim0.shape[1]
    
    cors_dict = {
        'joint': np.zeros(n_lags),
        'word': np.zeros(n_lags),
        'future': np.zeros(n_lags),
        'past': np.zeros(n_lags)
    }
    
    for lag in range(n_lags):
        cors_dict['joint'][lag] = pearsonr(dim0[:, lag], dim1[:, lag])[0]
        cors_dict['word'][lag] = pearsonr(dim0[:, lag], dim2[:, lag])[0]
        cors_dict['future'][lag] = pearsonr(dim0[:, lag], dim3[:, lag])[0]
        cors_dict['past'][lag] = pearsonr(dim0[:, lag], dim4[:, lag])[0]
    
    return cors_dict


In [ ]:
# plot all electrodes


# Track electrodes with unexpected lengths
weird_length_electrodes = []
# Create PDF for output
with PdfPages(output_pdf) as pdf:
    # loop over subjects (625, 676, 7170, 798)
    for sid in ["625", "676", "7170", "798"]:
        # loop over mode (comp, prod)
        for mode in ["comp", "prod"]:
            # load indices for first 2 last 2 words
            start_inds_f = f"{ind_d}/{sid}_{mode}_first2_indices.txt"
            start_inds = [int(line.strip()) for line in open(start_inds_f, "r").readlines()]
            end_inds_f = f"{ind_d}/{sid}_{mode}_last2_indices.txt"
            end_inds = [int(line.strip()) for line in open(end_inds_f, "r").readlines()]
            
            elecs_df = elecs_comp if mode == "comp" else elecs_prod
            ratio_col = "max_prediction_ratio" if mode == "comp" else "max_planning_ratio"
            sid_elecs_df = elecs_df[elecs_df['subject'] == int(sid)]
            
            # Sort electrodes by ratio column in descending order
            sid_elecs_df = sid_elecs_df.sort_values(ratio_col, ascending=False)
            
            expected_dim1 = max_dim1_dict[(sid, mode)]
            
            for idx, row in sid_elecs_df.iterrows():
                elec = row['electrode']
                roi = row.get('roi', 'N/A')
                max_joint = row.get('max_joint', 0.0)
                ratio_value = row.get(ratio_col, 0.0)
                
                preds_f = preds_d_tmp.format(sid=sid) + f"{sid}_{elec}_{mode}_predictions.h5"
                
                try:
                    with h5py.File(preds_f, "r") as hf:
                        elec_preds = hf['predictions'][:]
                    
                    # Check if dim1 matches expected value
                    actual_dim1 = elec_preds.shape[1]
                    if actual_dim1 != expected_dim1:
                        # Save metadata and skip plotting
                        weird_elec = row.to_dict()
                        weird_elec['sum_samples'] = actual_dim1
                        weird_elec['mode'] = mode
                        weird_length_electrodes.append(weird_elec)
                        print(f"Skipping {sid} {elec} {mode}: expected {expected_dim1}, got {actual_dim1}")
                        continue
                    
                    # Calculate correlations for full, start, and end across all lags
                    cors_full = calculate_correlations_per_lag(elec_preds)
                    cors_start = calculate_correlations_per_lag(elec_preds, start_inds)
                    cors_end = calculate_correlations_per_lag(elec_preds, end_inds)
                    
                    # Create figure with 4 subplots
                    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
                    
                    pred_types = ['joint', 'word', 'future', 'past']
                    
                    for i, pred_type in enumerate(pred_types):
                        ax = axes[i]
                        
                        # Get the color for this prediction type
                        color = pred_type_colors[pred_type]
                        
                        # Get correlations for this prediction type across all lags
                        full_cors = cors_full[pred_type]
                        start_cors = cors_start[pred_type]
                        end_cors = cors_end[pred_type]
                        
                        # Plot three lines:
                        # Full: grey dotted line
                        ax.plot(lags_s, full_cors, color='grey', linestyle=':', 
                               label='Full', linewidth=2, alpha=0.8)
                        # Start: solid colored line with alpha=1
                        ax.plot(lags_s, start_cors, color=color, linestyle='-', alpha=1.0,
                               label='Start', linewidth=2)
                        # End: solid colored line with alpha=0.5
                        ax.plot(lags_s, end_cors, color=color, linestyle='-', alpha=0.5,
                               label='End', linewidth=2)
                        
                        # Add vertical line at lag 0
                        ax.axvline(x=0, color='black', linestyle='--', alpha=0.3, linewidth=1)
                        ax.axhline(y=0, color='black', linestyle='-', alpha=0.2, linewidth=0.5)
                        
                        ax.set_xlabel('Lag (s)', fontsize=10)
                        ax.set_ylabel('Correlation', fontsize=10)
                        ax.set_title(pred_type.capitalize(), fontsize=12, fontweight='bold')
                        ax.set_ylim([0, 0.2])
                        ax.grid(True, alpha=0.3)
                        ax.legend(loc='upper right', fontsize=9)
                        
                        # Set x-axis ticks
                        ax.set_xlim([lags_s[0], lags_s[-1]])
                    
                    # Add metadata as suptitle
                    fig.suptitle(f'Subject: {sid} | Electrode: {elec} | Mode: {mode} | ROI: {roi} | '
                               f'Max Joint: {max_joint:.2f} | Max Ratio: {ratio_value:.2f}',
                               fontsize=13, y=1.00)
                    
                    plt.tight_layout()
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close(fig)
                    
                    print(f"Processed: {sid} {elec} {mode}")
                    
                except Exception as e:
                    print(f"Error processing {sid} {elec} {mode}: {e}")
                    continue

# Save weird length electrodes to CSV
if weird_length_electrodes:
    weird_df = pd.DataFrame(weird_length_electrodes)
    weird_df.to_csv(weird_lengths_csv, index=False)
    print(f"\nWeird length electrodes saved to: {weird_lengths_csv}")
    print(f"Total weird electrodes: {len(weird_length_electrodes)}")
else:
    print("\nNo electrodes with unexpected lengths found.")

print(f"PDF saved to: {output_pdf}")

In [5]:
# Sanity check: verify prediction dimensions for all files
sanity_check_data = []

for sid in ["625", "676", "7170", "798"]:
    for mode in ["comp", "prod"]:
        elecs_df = elecs_comp if mode == "comp" else elecs_prod
        sid_elecs_df = elecs_df[elecs_df['subject'] == int(sid)]
        
        for idx, row in sid_elecs_df.iterrows():
            elec = row['electrode']
            preds_f = preds_d_tmp.format(sid=sid) + f"{sid}_{elec}_{mode}_predictions.h5"
            
            try:
                with h5py.File(preds_f, "r") as hf:
                    elec_preds = hf['predictions'][:]
                    shape = elec_preds.shape
                    
                    sanity_check_data.append({
                        'subject': sid,
                        'mode': mode,
                        'electrode': elec,
                        'dim0': shape[0],
                        'dim1': shape[1],
                        'dim2': shape[2],
                        'file_exists': True
                    })
                    print(f"{sid} {elec} {mode}: shape = {shape}")
                    
            except Exception as e:
                sanity_check_data.append({
                    'subject': sid,
                    'mode': mode,
                    'electrode': elec,
                    'dim0': None,
                    'dim1': None,
                    'dim2': None,
                    'file_exists': False
                })
                print(f"Error loading {sid} {elec} {mode}: {e}")

sanity_df = pd.DataFrame(sanity_check_data)
print("\n=== Sanity Check Summary ===")
print(sanity_df.groupby(['subject', 'mode']).agg({
    'dim1': ['mean', 'min', 'max', 'std'],
    'file_exists': 'sum'
}))

625 EEGGR_38REF comp: shape = (5, 38176, 81)
625 EEGGR_28REF comp: shape = (5, 38176, 81)
625 EEGGR_63REF comp: shape = (5, 38176, 81)
625 EEGGR_54REF comp: shape = (5, 38176, 81)
625 EEGGR_62REF comp: shape = (5, 38176, 81)
625 EEGGR_26REF comp: shape = (5, 38176, 81)
625 EEGGR_20REF comp: shape = (5, 38176, 81)
625 EEGATO_01REF comp: shape = (5, 38176, 81)
625 EEGGR_55REF comp: shape = (5, 38176, 81)
625 EEGGR_45REF comp: shape = (5, 38176, 81)
625 EEGGR_46REF comp: shape = (5, 38176, 81)
625 EEGGR_13REF comp: shape = (5, 38176, 81)
625 EEGGR_59REF comp: shape = (5, 38176, 81)
625 EEGGR_53REF comp: shape = (5, 38176, 81)
625 EEGGR_04REF comp: shape = (5, 38176, 81)
625 EEGGR_37REF comp: shape = (5, 38176, 81)
625 EEGGR_61REF comp: shape = (5, 38176, 81)
625 EEGGR_29REF comp: shape = (5, 38176, 81)
625 EEGGR_22REF comp: shape = (5, 38176, 81)
625 EEGGR_43REF prod: shape = (5, 24511, 81)
625 EEGGR_63REF prod: shape = (5, 24511, 81)
625 EEGGR_01REF prod: shape = (5, 24511, 81)
625 EEGAT

In [6]:
for (subj, mode), group in sanity_df.groupby(['subject', 'mode']):
    print(f"\nSubject: {subj}, Mode: {mode}")
    print(group['dim1'].value_counts(dropna=False))


Subject: 625, Mode: comp
dim1
38176    19
Name: count, dtype: int64

Subject: 625, Mode: prod
dim1
24511    38
Name: count, dtype: int64

Subject: 676, Mode: comp
dim1
91202    37
90843     3
Name: count, dtype: int64

Subject: 676, Mode: prod
dim1
92442    70
92104     3
44883     1
89934     1
70806     1
7861      1
88852     1
Name: count, dtype: int64

Subject: 7170, Mode: comp
dim1
58262    23
58869    21
57846     3
36334     3
56100     1
25959     1
Name: count, dtype: int64

Subject: 7170, Mode: prod
dim1
38477    42
38564    21
38402     6
16706     4
33534     1
12533     1
16793     1
Name: count, dtype: int64

Subject: 798, Mode: comp
dim1
50308    47
Name: count, dtype: int64

Subject: 798, Mode: prod
dim1
40869    36
Name: count, dtype: int64


In [ ]:
import sys
sys.path.append('/scratch/gpfs/HASSON/ij9216/projects/code/247/247-plotting/scripts')
import tfsplt_future_past_utils as pu

# Calculate differences between start and end encoding for future and past
# For future: use lags from -2000ms to 0ms
# For past: use lags from 0ms to 2000ms

future_diffs = []
past_diffs = []

for sid in ["625", "676", "7170", "798"]:
    for mode in ["comp", "prod"]:
        # Load indices
        start_inds_f = f"{ind_d}/{sid}_{mode}_first2_indices.txt"
        start_inds = [int(line.strip()) for line in open(start_inds_f, "r").readlines()]
        end_inds_f = f"{ind_d}/{sid}_{mode}_last2_indices.txt"
        end_inds = [int(line.strip()) for line in open(end_inds_f, "r").readlines()]
        
        elecs_df = elecs_comp if mode == "comp" else elecs_prod
        sid_elecs_df = elecs_df[elecs_df['subject'] == int(sid)]
        
        expected_dim1 = max_dim1_dict[(sid, mode)]
        
        for idx, row in sid_elecs_df.iterrows():
            elec = row['electrode']
            roi = row.get('roi', 'N/A')
            
            preds_f = preds_d_tmp.format(sid=sid) + f"{sid}_{elec}_{mode}_predictions.h5"
            
            try:
                with h5py.File(preds_f, "r") as hf:
                    elec_preds = hf['predictions'][:]
                
                # Check if dim1 matches expected value
                actual_dim1 = elec_preds.shape[1]
                if actual_dim1 != expected_dim1:
                    continue
                
                # Calculate correlations
                cors_start = calculate_correlations_per_lag(elec_preds, start_inds)
                cors_end = calculate_correlations_per_lag(elec_preds, end_inds)
                
                # Get lag indices for future (-2000ms to 0ms) and past (0ms to 2000ms)
                future_lag_mask = (lags_ms >= -2000) & (lags_ms <= 0)
                past_lag_mask = (lags_ms >= 0) & (lags_ms <= 2000)
                
                # Future: get max difference in the -2000 to 0 window
                future_start_vals = np.array(cors_start['future'])[future_lag_mask]
                future_end_vals = np.array(cors_end['future'])[future_lag_mask]
                # Set negative correlations to 0
                future_start_vals = np.maximum(future_start_vals, 0)
                future_end_vals = np.maximum(future_end_vals, 0)
                # Calculate max difference
                # max_future_diff = np.max(future_start_vals) - np.max(future_end_vals)
                future_diffs_per_lag = future_start_vals - future_end_vals
                max_abs_idx = np.argmax(np.abs(future_diffs_per_lag))
                max_future_diff = future_diffs_per_lag[max_abs_idx]


                
                future_diffs.append({
                    'subject': sid,
                    'electrode': elec,
                    'roi': roi,
                    'mode': mode,
                    'max_diff': max_future_diff
                })
                
                # Past: get max difference in the 0 to 2000 window
                past_start_vals = np.array(cors_start['past'])[past_lag_mask]
                past_end_vals = np.array(cors_end['past'])[past_lag_mask]
                # Set negative correlations to 0
                past_start_vals = np.maximum(past_start_vals, 0)
                past_end_vals = np.maximum(past_end_vals, 0)
                # Calculate max difference
                # max_past_diff = np.max(past_start_vals) - np.max(past_end_vals)
                past_diffs_per_lag = past_start_vals - past_end_vals
                max_abs_idx = np.argmax(np.abs(past_diffs_per_lag))
                max_past_diff = past_diffs_per_lag[max_abs_idx]
                
                past_diffs.append({
                    'subject': sid,
                    'electrode': elec,
                    'roi': roi,
                    'mode': mode,
                    'max_diff': max_past_diff
                })
                
            except Exception as e:
                print(f"Error processing {sid} {elec} {mode}: {e}")
                continue

# Convert to DataFrames
future_diff_df = pd.DataFrame(future_diffs)
past_diff_df = pd.DataFrame(past_diffs)

print(f"\nCalculated differences for:")
print(f"  Future: {len(future_diff_df)} electrodes")
print(f"  Past: {len(past_diff_df)} electrodes")

# Split by mode
future_comp = future_diff_df[future_diff_df['mode'] == 'comp']
future_prod = future_diff_df[future_diff_df['mode'] == 'prod']
past_comp = past_diff_df[past_diff_df['mode'] == 'comp']
past_prod = past_diff_df[past_diff_df['mode'] == 'prod']

# Plot brain maps
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Comprehension Future
pu.plot_effect_glassbrain(
    future_comp,
    effect_col='max_diff',
    cmap='bwr',
    vmin=-0.1,
    vmax=0.1,
    ax=axes[0, 0],
    title='Comprehension Future\n(Start - End, -2000ms to 0ms)',
    colorbar=True
)

# Comprehension Past
pu.plot_effect_glassbrain(
    past_comp,
    effect_col='max_diff',
    cmap='bwr',
    vmin=-0.1,
    vmax=0.1,
    ax=axes[0, 1],
    title='Comprehension Past\n(Start - End, 0ms to 2000ms)',
    colorbar=True
)

# Production Future
pu.plot_effect_glassbrain(
    future_prod,
    effect_col='max_diff',
    cmap='bwr',
    vmin=-0.1,
    vmax=0.1,
    ax=axes[1, 0],
    title='Production Future\n(Start - End, -2000ms to 0ms)',
    colorbar=True
)

# Production Past
pu.plot_effect_glassbrain(
    past_prod,
    effect_col='max_diff',
    cmap='bwr',
    vmin=-0.1,
    vmax=0.1,
    ax=axes[1, 1],
    title='Production Past\n(Start - End, 0ms to 2000ms)',
    colorbar=True
)

plt.tight_layout()
plt.savefig(f"{ind_d}/sentence_start_end_brain_differences.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBrain map saved to: {ind_d}/sentence_start_end_brain_differences.png")

# Print summary statistics
print("\n=== Summary Statistics ===")
print("\nFuture (Comprehension):")
print(f"  Mean difference: {future_comp['max_diff'].mean():.4f}")
print(f"  Median difference: {future_comp['max_diff'].median():.4f}")
print(f"  Positive differences: {(future_comp['max_diff'] > 0).sum()} / {len(future_comp)}")

print("\nFuture (Production):")
print(f"  Mean difference: {future_prod['max_diff'].mean():.4f}")
print(f"  Median difference: {future_prod['max_diff'].median():.4f}")
print(f"  Positive differences: {(future_prod['max_diff'] > 0).sum()} / {len(future_prod)}")

print("\nPast (Comprehension):")
print(f"  Mean difference: {past_comp['max_diff'].mean():.4f}")
print(f"  Median difference: {past_comp['max_diff'].median():.4f}")
print(f"  Positive differences: {(past_comp['max_diff'] > 0).sum()} / {len(past_comp)}")

print("\nPast (Production):")
print(f"  Mean difference: {past_prod['max_diff'].mean():.4f}")
print(f"  Median difference: {past_prod['max_diff'].median():.4f}")
print(f"  Positive differences: {(past_prod['max_diff'] > 0).sum()} / {len(past_prod)}")